# Benchmark upload to Google Drive

Downloads and verifies the requested visualization/NL2BI benchmarks into Google Drive. The Drive layout is organized under `My Drive/diploma/petr_text_to_visualization_part` with separate folders for benchmark artifacts, notebooks, and reports.

In [ ]:
%pip -q install -U huggingface_hub datasets gdown tqdm pandas requests
print('[setup] dependencies installed')


In [2]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    if not Path('/content/drive/MyDrive').exists():
        raise RuntimeError('Google Drive is not mounted. Open this notebook in Colab and complete Drive authorization.') from exc

DIPLOMA_ROOT = Path('/content/drive/MyDrive/diploma')
MY_PART_ROOT = Path(os.environ.get('MY_PART_ROOT', str(DIPLOMA_ROOT / 'petr_text_to_visualization_part')))
DRIVE_ROOT = Path(os.environ.get('BENCHMARK_DRIVE_ROOT', str(MY_PART_ROOT / 'benchmarks')))
NOTEBOOK_ROOT = MY_PART_ROOT / 'notebooks'
REPORT_ROOT = MY_PART_ROOT / 'reports'

for folder in [DIPLOMA_ROOT, MY_PART_ROOT, DRIVE_ROOT, NOTEBOOK_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'[setup] DIPLOMA_ROOT={DIPLOMA_ROOT}')
print(f'[setup] MY_PART_ROOT={MY_PART_ROOT}')
print(f'[setup] DRIVE_ROOT={DRIVE_ROOT}')
print(f'[setup] NOTEBOOK_ROOT={NOTEBOOK_ROOT}')
print(f'[setup] REPORT_ROOT={REPORT_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[setup] DIPLOMA_ROOT=/content/drive/MyDrive/diploma
[setup] MY_PART_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part
[setup] DRIVE_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part/benchmarks
[setup] NOTEBOOK_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part/notebooks
[setup] REPORT_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part/reports


In [3]:
from pathlib import Path
import shutil

mydrive = Path('/content/drive/MyDrive')
diploma_root = mydrive / 'diploma'
my_part_root = diploma_root / 'petr_text_to_visualization_part'
benchmarks_root = my_part_root / 'benchmarks'
notebooks_root = my_part_root / 'notebooks'
reports_root = my_part_root / 'reports'

for folder in [diploma_root, my_part_root, benchmarks_root, notebooks_root, reports_root]:
    folder.mkdir(parents=True, exist_ok=True)


def merge_tree(src: Path, dst: Path) -> None:
    if not src.exists():
        print(f'[ok] source not present: {src}')
        return
    if src.resolve() == dst.resolve():
        print(f'[ok] source already target: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    moved = 0
    merged = 0
    skipped = 0
    for child in src.iterdir():
        target = dst / child.name
        if target.exists() and child.is_dir() and target.is_dir():
            merge_tree(child, target)
            merged += 1
            continue
        if target.exists():
            print(f'[skip-existing] {target}')
            skipped += 1
            continue
        shutil.move(str(child), str(target))
        moved += 1
    try:
        src.rmdir()
        print(f'[remove-empty] {src}')
    except OSError:
        print(f'[kept] source not empty after merge: {src}')
    print(f'[merge-result] {src.name}: moved={moved}; merged={merged}; skipped={skipped}')

# Rename the previous generic folder into the explicit Peter text-to-visualization part.
merge_tree(diploma_root / 'my_part', my_part_root)
# Old location from the first upload pass.
merge_tree(diploma_root / 'nl2bi_benchmarks', benchmarks_root)
# Safety for the pre-diploma location if it ever reappears.
merge_tree(mydrive / 'nl2bi_benchmarks', benchmarks_root)

report_names = {
    'README.md',
    'download_manifest.json',
    'download_manifest.partial.json',
    'download_manifest.csv',
    'source_preflight.json',
    'source_preflight.csv',
}
for name in sorted(report_names):
    src = benchmarks_root / name
    dst = reports_root / name
    if src.exists() and not dst.exists():
        shutil.move(str(src), str(dst))
        print(f'[report] {src} -> {dst}')
    elif src.exists() and dst.exists():
        print(f'[report-skip-existing] {dst}')

# Keep these globals aligned for subsequent cells.
DIPLOMA_ROOT = diploma_root
MY_PART_ROOT = my_part_root
DRIVE_ROOT = benchmarks_root
NOTEBOOK_ROOT = notebooks_root
REPORT_ROOT = reports_root
print(f'[done] MY_PART_ROOT={MY_PART_ROOT}')
print(f'[done] DRIVE_ROOT={DRIVE_ROOT}')
print(f'[done] NOTEBOOK_ROOT={NOTEBOOK_ROOT}')
print(f'[done] REPORT_ROOT={REPORT_ROOT}')

[remove-empty] /content/drive/MyDrive/diploma/my_part/benchmarks
[merge-result] benchmarks: moved=9; merged=0; skipped=0
[remove-empty] /content/drive/MyDrive/diploma/my_part/notebooks
[merge-result] notebooks: moved=1; merged=0; skipped=0
[remove-empty] /content/drive/MyDrive/diploma/my_part/reports
[merge-result] reports: moved=6; merged=0; skipped=0
[remove-empty] /content/drive/MyDrive/diploma/my_part
[merge-result] my_part: moved=0; merged=3; skipped=0
[ok] source not present: /content/drive/MyDrive/diploma/nl2bi_benchmarks
[ok] source not present: /content/drive/MyDrive/nl2bi_benchmarks
[done] MY_PART_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part
[done] DRIVE_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part/benchmarks
[done] NOTEBOOK_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part/notebooks
[done] REPORT_ROOT=/content/drive/MyDrive/diploma/petr_text_to_visualization_part/reports


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import tarfile
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import gdown
import pandas as pd
import requests
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

REQUEST_TIMEOUT = 60
CHUNK_SIZE = 1024 * 1024
MAX_SINGLE_DOWNLOAD_BYTES = int(os.environ.get('MAX_SINGLE_DOWNLOAD_BYTES', str(30 * 1024**3)))
DOWNLOAD_VIZML_FULL = os.environ.get('DOWNLOAD_VIZML_FULL', '0') == '1'
DOWNLOAD_VIZML_FEATURES = os.environ.get('DOWNLOAD_VIZML_FEATURES', '0') == '1'

BENCHMARKS = [
    {'name': 'VisEval', 'method': 'github_archive', 'repo': 'microsoft/VisEval', 'branch': 'main', 'required': True, 'source': 'https://github.com/microsoft/VisEval', 'note': 'Repo includes viseval_dataset.zip used by the benchmark.'},
    {'name': 'Text2Vis', 'method': 'hf_dataset', 'repo_id': 'mizanurr/Text2Vis', 'required': True, 'source': 'https://huggingface.co/datasets/mizanurr/Text2Vis'},
    {'name': 'NVBench', 'method': 'github_archive', 'repo': 'TsinghuaDatabaseGroup/nvBench', 'branch': 'main', 'required': True, 'source': 'https://github.com/TsinghuaDatabaseGroup/nvBench', 'note': 'Includes NVBench.json, databases.zip, and Vega-Lite files.'},
    {'name': 'nvBench 2.0', 'method': 'hf_dataset', 'repo_id': 'TianqiLuo/nvBench2.0', 'required': True, 'source': 'https://huggingface.co/datasets/TianqiLuo/nvBench2.0'},
    {'name': 'ChartDialogs', 'method': 'gdrive_file', 'file_id': '1B8tKdmI3LlAMsPQQW6KsCbTKfsm9vnFA', 'filename': 'ChartDialog-data.zip', 'required': True, 'source': 'https://github.com/sythello/ChartDialog', 'note': 'Dataset link is published in the ChartDialog README.'},
    {'name': 'VisJudge-Bench', 'method': 'github_archive', 'repo': 'HKUSTDial/VisJudgeBench', 'branch': 'main', 'required': True, 'source': 'https://github.com/HKUSTDial/VisJudgeBench', 'note': 'Includes VisJudgeBench.json and image assets.'},
    {'name': 'Waltzboard', 'method': 'github_archive', 'repo': 'jiwnchoi/Waltzboard', 'branch': 'main', 'required': True, 'source': 'https://github.com/jiwnchoi/Waltzboard', 'note': 'Public repo contains the packaged examples and data directory.'},
    {'name': 'VizML', 'method': 'not_available', 'required': False, 'source': 'https://github.com/mitmedialab/vizml', 'reason': 'Official VizML S3 corpus URLs from retrieve_data.sh return 403 Forbidden for 1K, 100K, full/features artifacts; benchmark corpus cannot be downloaded from the public source.'},
    {'name': 'DMiner', 'method': 'gdrive_folder', 'folder_url': 'https://drive.google.com/drive/folders/1ECQwC9kdfY_xim6N9g4m9Rm2Nrrwk4az?usp=sharing', 'required': True, 'source': 'https://yannahhh.github.io/publications/', 'note': 'Authors supplementary folder for Dashboard Design Mining and Recommendation.'},
    {'name': 'DashBot', 'method': 'not_available', 'required': False, 'source': 'https://arxiv.org/abs/2208.01232', 'reason': 'No public downloadable benchmark/dataset artifact was found; DashBot is excluded from required downloads.'},
]

@dataclass
class Result:
    name: str
    status: str
    path: str
    source: str
    method: str
    required: bool
    file_count: int = 0
    total_bytes: int = 0
    elapsed_seconds: float = 0.0
    note: str = ''
    error: str = ''


def safe_name(name: str) -> str:
    return ''.join(ch if ch.isalnum() else '_' for ch in name).strip('_')


def dir_stats(path: Path) -> tuple[int, int]:
    if not path.exists():
        return 0, 0
    count = 0
    total = 0
    for item in path.rglob('*'):
        if item.is_file():
            count += 1
            try:
                total += item.stat().st_size
            except OSError:
                pass
    return count, total


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=True, indent=2), encoding='utf-8')


def download_file(url: str, dest: Path, max_bytes: int | None = MAX_SINGLE_DOWNLOAD_BYTES) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    headers = {'User-Agent': 'nl2bi-benchmark-upload/1.0'}
    with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT, headers=headers) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length') or 0)
        if max_bytes is not None and total and total > max_bytes:
            raise RuntimeError(f'Refusing to download {total} bytes from {url}; limit is {max_bytes}.')
        if dest.exists() and total and dest.stat().st_size == total:
            print(f'[skip] {dest.name} already exists ({total} bytes)')
            return dest
        tmp = dest.with_suffix(dest.suffix + '.part')
        with tmp.open('wb') as handle, tqdm(total=total or None, unit='B', unit_scale=True, desc=dest.name) as bar:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    handle.write(chunk)
                    bar.update(len(chunk))
        tmp.replace(dest)
    return dest


def extract_archive(archive: Path, target: Path) -> None:
    marker = target / '.extract_complete'
    if marker.exists():
        print(f'[skip] extracted {archive.name}')
        return
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as tf:
            tf.extractall(target)
    else:
        raise RuntimeError(f'Unsupported archive format: {archive}')
    marker.write_text(time.strftime('%Y-%m-%dT%H:%M:%S'), encoding='utf-8')


def download_github_archive(spec: dict[str, Any], target: Path) -> None:
    repo = spec['repo']
    branch = spec.get('branch', 'main')
    archive = target / f'{safe_name(repo)}-{branch}.zip'
    extracted = target / 'repo'
    url = f'https://github.com/{repo}/archive/refs/heads/{branch}.zip'
    download_file(url, archive)
    extract_archive(archive, extracted)


def download_hf_dataset(spec: dict[str, Any], target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=spec['repo_id'], repo_type='dataset', local_dir=str(target), local_dir_use_symlinks=False, resume_download=True)


def download_gdrive_file(spec: dict[str, Any], target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    existing_files, existing_total = dir_stats(target)
    if existing_files > 0 and existing_total > 1024 * 1024:
        print(f'[skip] existing Google Drive artifact data in {target}: files={existing_files}; bytes={existing_total}')
        return
    dest = target / spec['filename']
    if dest.exists() and dest.stat().st_size > 0:
        print(f'[skip] {dest} already exists')
    else:
        url = f'https://drive.google.com/uc?id={spec["file_id"]}'
        output = gdown.download(url, str(dest), quiet=False)
        if not output or not dest.exists() or dest.stat().st_size == 0:
            raise RuntimeError(f'gdown failed to download {spec["file_id"]}')
    # Keep public Google Drive archives compressed on Drive. Extracting large archives
    # directly into mounted Drive is slow and can stall Colab sessions; the archive
    # itself is the verified downloaded artifact.


def download_gdrive_folder(spec: dict[str, Any], target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    marker = target / '.gdrive_folder_complete'
    if marker.exists():
        print(f'[skip] Google Drive folder already downloaded: {target}')
        return
    gdown.download_folder(url=spec['folder_url'], output=str(target), quiet=False, use_cookies=False)
    files, total = dir_stats(target)
    if files == 0 or total == 0:
        raise RuntimeError(f'gdown folder download produced no files: {spec["folder_url"]}')
    marker.write_text(time.strftime('%Y-%m-%dT%H:%M:%S'), encoding='utf-8')


def download_vizml(spec: dict[str, Any], target: Path) -> None:
    download_github_archive(spec, target / 'code')
    data_dir = target / 'data'
    data_dir.mkdir(parents=True, exist_ok=True)
    base = 'http://vizml-repository.s3.amazonaws.com'
    files = ['plotly_subset_100k.tar.gz']
    if DOWNLOAD_VIZML_FEATURES:
        files.append('features.tar.gz')
    if DOWNLOAD_VIZML_FULL:
        files.append('plotly_full.tar.gz')
    for filename in files:
        archive = download_file(f'{base}/{filename}', data_dir / filename)
        extract_archive(archive, data_dir / filename.replace('.tar.gz', ''))


def process_benchmark(spec: dict[str, Any]) -> Result:
    name = spec['name']
    method = spec['method']
    target = DRIVE_ROOT / safe_name(name)
    started = time.time()
    print(f'\n=== {name} ({method}) ===')
    try:
        if method == 'not_available':
            return Result(name, 'not_available', str(target), spec.get('source', ''), method, False, 0, 0, round(time.time() - started, 3), spec.get('reason', ''))
        if method == 'github_archive':
            download_github_archive(spec, target)
        elif method == 'hf_dataset':
            download_hf_dataset(spec, target)
        elif method == 'gdrive_file':
            download_gdrive_file(spec, target)
        elif method == 'gdrive_folder':
            download_gdrive_folder(spec, target)
        elif method == 'vizml':
            download_vizml(spec, target)
        else:
            raise RuntimeError(f'Unknown method: {method}')
        files, total = dir_stats(target)
        status = 'downloaded' if files > 0 and total > 0 else 'empty'
        return Result(name, status, str(target), spec.get('source', ''), method, bool(spec.get('required', False)), files, total, round(time.time() - started, 3), spec.get('note', ''))
    except Exception as exc:
        files, total = dir_stats(target)
        return Result(name, 'blocked' if spec.get('required', False) else 'failed', str(target), spec.get('source', ''), method, bool(spec.get('required', False)), files, total, round(time.time() - started, 3), spec.get('note', ''), str(exc))

print(f'[manifest] {len(BENCHMARKS)} benchmarks configured')
print(f'[policy] MAX_SINGLE_DOWNLOAD_BYTES={MAX_SINGLE_DOWNLOAD_BYTES}')
print(f'[policy] DOWNLOAD_VIZML_FULL={DOWNLOAD_VIZML_FULL}; DOWNLOAD_VIZML_FEATURES={DOWNLOAD_VIZML_FEATURES}')


In [ ]:
source_rows = []
blocking_sources = []
for spec in BENCHMARKS:
    if spec.get('method') in {'not_available', 'manual_blocker'}:
        row = {'name': spec['name'], 'status': 'not_available', 'required': bool(spec.get('required', False)), 'source': spec.get('source', ''), 'reason': spec.get('reason', '')}
        source_rows.append(row)
        if spec.get('required', False):
            blocking_sources.append(row)

preflight_path = REPORT_ROOT / 'source_preflight.json'
write_json(preflight_path, source_rows)
pd.DataFrame(source_rows).to_csv(REPORT_ROOT / 'source_preflight.csv', index=False)

if blocking_sources:
    print('[UPLOAD_BENCHMARKS_BLOCKER] Required benchmarks have no public downloadable artifact:')
    for item in blocking_sources:
        print(f"- {item['name']}: {item['reason']} ({item['source']})")
    print(f'[preflight] saved to {preflight_path}')
    raise RuntimeError('Blocking source preflight failed. Add public/manual download sources for the listed benchmarks before running downloads.')

if source_rows:
    print('[preflight] optional unavailable benchmarks:')
    for item in source_rows:
        print(f"- {item['name']}: {item['reason']} ({item['source']})")
print(f'[preflight] saved to {preflight_path}')
print('[preflight] all required benchmarks have downloadable sources')


In [ ]:
results = []
for spec in BENCHMARKS:
    result = process_benchmark(spec)
    results.append(result)
    print(f'[result] {result.name}: {result.status}; files={result.file_count}; bytes={result.total_bytes}; error={result.error[:160]}')
    write_json(REPORT_ROOT / 'download_manifest.partial.json', [asdict(item) for item in results])

summary = [asdict(item) for item in results]
write_json(REPORT_ROOT / 'download_manifest.json', summary)
pd.DataFrame(summary).to_csv(REPORT_ROOT / 'download_manifest.csv', index=False)
print(f'[manifest] saved to {REPORT_ROOT / "download_manifest.json"}')


In [ ]:
from IPython.display import display

summary = [asdict(item) for item in results]
df = pd.DataFrame(summary)
display(df[['name', 'status', 'required', 'file_count', 'total_bytes', 'elapsed_seconds', 'error']])

readme_lines = [
    '# NL2BI benchmark downloads',
    '',
    f'Drive root: `{DRIVE_ROOT}`',
    '',
    '| Benchmark | Required | Status | Files | Bytes | Source | Note/Error |',
    '|---|---:|---:|---:|---:|---|---|',
]
for item in summary:
    note = item.get('error') or item.get('note') or ''
    note = note.replace('|', '/')
    readme_lines.append(f"| {item['name']} | {item['required']} | {item['status']} | {item['file_count']} | {item['total_bytes']} | {item['source']} | {note} |")
(REPORT_ROOT / 'README.md').write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

blocked = [item for item in summary if item['required'] and item['status'] != 'downloaded']
if blocked:
    print('[UPLOAD_BENCHMARKS_BLOCKER] Required benchmarks were not fully downloaded:')
    for item in blocked:
        print(f"- {item['name']}: {item['error'] or item['status']} ({item['source']})")
    raise RuntimeError('Blocking benchmark download issues remain. See download_manifest.json on Google Drive.')

print('[UPLOAD_BENCHMARKS_COMPLETED] All required benchmarks downloaded and verified. Optional unavailable entries are recorded in the manifest.')


In [4]:
import base64
from pathlib import Path

NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)
notebook_bytes = base64.b64decode('...')
target = NOTEBOOK_ROOT / 'upload.ipynb'
target.write_bytes(notebook_bytes)
print(f'[notebook] wrote {target} ({len(notebook_bytes)} bytes)')


[notebook] wrote /content/drive/MyDrive/diploma/petr_text_to_visualization_part/notebooks/upload.ipynb (44085 bytes)


In [5]:
from pathlib import Path

paths = [
    Path('/content/drive/MyDrive/diploma'),
    Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part'),
    Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part/benchmarks'),
    Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part/notebooks'),
    Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part/reports'),
    Path('/content/drive/MyDrive/diploma/my_part'),
    Path('/content/drive/MyDrive/diploma/nl2bi_benchmarks'),
]
for path in paths:
    print(f'PATH {path} exists={path.exists()}')
    if path.exists():
        items = sorted(p.name for p in path.iterdir())
        print('  items:', items[:30], 'count=', len(items))

PATH /content/drive/MyDrive/diploma exists=True
  items: ['diploma_plan_sql', 'petr_text_to_visualization_part'] count= 2
PATH /content/drive/MyDrive/diploma/petr_text_to_visualization_part exists=True
  items: ['benchmarks', 'notebooks', 'reports'] count= 3
PATH /content/drive/MyDrive/diploma/petr_text_to_visualization_part/benchmarks exists=True
  items: ['ChartDialogs', 'DMiner', 'NVBench', 'Text2Vis', 'VisEval', 'VisJudge_Bench', 'VizML', 'Waltzboard', 'nvBench_2_0'] count= 9
PATH /content/drive/MyDrive/diploma/petr_text_to_visualization_part/notebooks exists=True
  items: ['upload.ipynb'] count= 1
PATH /content/drive/MyDrive/diploma/petr_text_to_visualization_part/reports exists=True
  items: ['README.md', 'download_manifest.csv', 'download_manifest.json', 'download_manifest.partial.json', 'source_preflight.csv', 'source_preflight.json'] count= 6
PATH /content/drive/MyDrive/diploma/my_part exists=False
PATH /content/drive/MyDrive/diploma/nl2bi_benchmarks exists=False
